# Open Food Facts ETL — phân tích mart

**Mục tiêu:** trả lời 6 câu hỏi phân tích từ các bảng trong schema `mart` (PostgreSQL), không đọc `core` / `staging` / `raw`.

**Nguồn dữ liệu:** [Open Food Facts](https://world.openfoodfacts.org/) — dump sản phẩm thực phẩm. Giấy phép dữ liệu: **ODbL** (Open Database License).

**Sáu câu hỏi:**
1. Nutri-Score theo quốc gia — phân bố grade a–e ở 10 nước có nhiều sản phẩm được chấm điểm nhất.
2. Danh mục nhiều đường — 15 danh mục có trung vị đường (g/100g) cao nhất.
3. NOVA vs Nutri-Score — phân bố Nutri-Score trong từng nhóm NOVA.
4. Thương hiệu — 10 thương hiệu tỷ lệ Nutri-Score A/B cao nhất và 10 thấp nhất.
5. Độ đầy đủ dữ liệu theo quốc gia — Nutri-Score, dinh dưỡng, NOVA, danh mục.
6. Theo thời gian — số sản phẩm theo năm tạo và tỷ lệ có Nutri-Score.

In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

sys.path.append("../src")
from config import PG, ROOT

url = URL.create(
    "postgresql+psycopg2",
    username=PG["user"],
    password=PG["password"],
    host=PG["host"],
    port=PG["port"],
    database=PG["dbname"],
)
engine = create_engine(url)


def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, engine)


img_dir = ROOT / "docs" / "img"
img_dir.mkdir(parents=True, exist_ok=True)

NUTRISCORE_COLORS = {
    "a": "#038141",
    "b": "#85BB2F",
    "c": "#FECB02",
    "d": "#EE8100",
    "e": "#E63E11",
}
GRADES = ["a", "b", "c", "d", "e"]

## 1. Nutri-Score theo quốc gia

Lấy 10 nước có `n_scored` lớn nhất từ `mart.country_nutriscore`. Biểu đồ cột chồng 100% các grade a–e (tỷ lệ `n_a`…`n_e` / `n_scored`), màu từ xanh đến đỏ.

In [ ]:
df_country_ns = q(
    """
    SELECT
        country_name,
        n_scored,
        n_a,
        n_b,
        n_c,
        n_d,
        n_e
    FROM mart.country_nutriscore
    ORDER BY n_scored DESC
    LIMIT 10
    """
)

share = df_country_ns.copy()
for g in GRADES:
    share[g] = share[f"n_{g}"] / share["n_scored"]

fig, ax = plt.subplots(figsize=(10, 6))
bottom = [0.0] * len(share)
x = range(len(share))
for g in GRADES:
    ax.bar(
        x,
        share[g],
        bottom=bottom,
        color=NUTRISCORE_COLORS[g],
        label=g.upper(),
    )
    bottom = [b + v for b, v in zip(bottom, share[g])]

ax.set_xticks(list(x))
ax.set_xticklabels(share["country_name"], rotation=45, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Tỷ lệ trong sản phẩm đã chấm điểm")
ax.set_xlabel("Quốc gia")
ax.set_title("Phân bố Nutri-Score (A–E) — 10 nước có n_scored lớn nhất")
ax.legend(title="Grade", loc="upper left", bbox_to_anchor=(1.02, 1))
fig.tight_layout()
fig.savefig(img_dir / "country_nutriscore.png", dpi=150)
plt.show()

### Insight


## 2. Danh mục nhiều đường

15 danh mục có `median_sugars_100g` cao nhất trong `mart.category_nutrition` (`n_products >= 1000`). Biểu đồ cột ngang theo trung vị đường (g/100g).

In [ ]:
df_sugar = q(
    """
    SELECT
        category_name,
        median_sugars_100g,
        n_products
    FROM mart.category_nutrition
    WHERE n_products >= 1000
    ORDER BY median_sugars_100g DESC
    LIMIT 15
    """
)

plot_sugar = df_sugar.sort_values("median_sugars_100g", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(plot_sugar["category_name"], plot_sugar["median_sugars_100g"], color="#C44E52")
ax.set_xlabel("Trung vị đường (g/100g)")
ax.set_ylabel("Danh mục")
ax.set_title("15 danh mục có trung vị đường cao nhất (n_products ≥ 1000)")
fig.tight_layout()
fig.savefig(img_dir / "category_sugars.png", dpi=150)
plt.show()

### Insight


## 3. NOVA vs Nutri-Score

Từ `mart.nova_vs_nutriscore`: biểu đồ cột chồng 100% theo `nova_group`, các grade a–e.

In [ ]:
df_nova = q(
    """
    SELECT
        nova_group,
        nutriscore_grade_clean,
        n_products
    FROM mart.nova_vs_nutriscore
    """
)

pivot = (
    df_nova.pivot_table(
        index="nova_group",
        columns="nutriscore_grade_clean",
        values="n_products",
        aggfunc="sum",
        fill_value=0,
    )
    .reindex(columns=GRADES, fill_value=0)
    .sort_index()
)
share_nova = pivot.div(pivot.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(8, 6))
bottom = [0.0] * len(share_nova)
x = range(len(share_nova))
for g in GRADES:
    ax.bar(
        x,
        share_nova[g],
        bottom=bottom,
        color=NUTRISCORE_COLORS[g],
        label=g.upper(),
    )
    bottom = [b + v for b, v in zip(bottom, share_nova[g])]

ax.set_xticks(list(x))
ax.set_xticklabels([f"NOVA {int(v)}" for v in share_nova.index])
ax.set_ylim(0, 1)
ax.set_ylabel("Tỷ lệ trong nhóm NOVA")
ax.set_xlabel("Nhóm NOVA")
ax.set_title("Phân bố Nutri-Score (A–E) theo nhóm NOVA")
ax.legend(title="Grade", loc="upper left", bbox_to_anchor=(1.02, 1))
fig.tight_layout()
fig.savefig(img_dir / "nova_vs_nutriscore.png", dpi=150)
plt.show()

### Insight


## 4. Thương hiệu

Từ `mart.brand_scores`: 10 thương hiệu có `pct_a_b` cao nhất và 10 thấp nhất (cột ngang).

In [ ]:
df_brand = q(
    """
    SELECT
        brand,
        pct_a_b,
        n_scored,
        n_products
    FROM mart.brand_scores
    """
)

top10 = df_brand.nlargest(10, "pct_a_b").sort_values("pct_a_b", ascending=True)
bot10 = df_brand.nsmallest(10, "pct_a_b").sort_values("pct_a_b", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=False)

axes[0].barh(top10["brand"], top10["pct_a_b"], color="#038141")
axes[0].set_xlabel("% Nutri-Score A hoặc B")
axes[0].set_ylabel("Thương hiệu")
axes[0].set_title("10 thương hiệu có pct_a_b cao nhất")

axes[1].barh(bot10["brand"], bot10["pct_a_b"], color="#E63E11")
axes[1].set_xlabel("% Nutri-Score A hoặc B")
axes[1].set_ylabel("Thương hiệu")
axes[1].set_title("10 thương hiệu có pct_a_b thấp nhất")

fig.suptitle("Tỷ lệ Nutri-Score A/B theo thương hiệu")
fig.tight_layout()
fig.savefig(img_dir / "brand_scores.png", dpi=150)
plt.show()

### Insight


## 5. Độ đầy đủ dữ liệu

15 nước có `n_products` lớn nhất trong `mart.country_completeness`. Biểu đồ cột nhóm cho `pct_with_nutriscore`, `pct_with_nutrition`, `pct_with_nova`, `pct_with_category`.

In [ ]:
df_comp = q(
    """
    SELECT
        country_name,
        n_products,
        pct_with_nutriscore,
        pct_with_nutrition,
        pct_with_nova,
        pct_with_category
    FROM mart.country_completeness
    ORDER BY n_products DESC
    LIMIT 15
    """
)

metrics = [
    "pct_with_nutriscore",
    "pct_with_nutrition",
    "pct_with_nova",
    "pct_with_category",
]
labels = [
    "% có Nutri-Score",
    "% có dinh dưỡng",
    "% có NOVA",
    "% có danh mục",
]
n = len(df_comp)
x = range(n)
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))
for i, (col, lab) in enumerate(zip(metrics, labels)):
    offset = (i - 1.5) * width
    ax.bar([xi + offset for xi in x], df_comp[col], width=width, label=lab)

ax.set_xticks(list(x))
ax.set_xticklabels(df_comp["country_name"], rotation=45, ha="right")
ax.set_ylabel("% sản phẩm")
ax.set_xlabel("Quốc gia")
ax.set_title("Độ đầy đủ dữ liệu — 15 nước có n_products lớn nhất")
ax.legend()
fig.tight_layout()
fig.savefig(img_dir / "country_completeness.png", dpi=150)
plt.show()

### Insight


## 6. Theo thời gian

Từ `mart.products_by_year`: cột `n_products` theo `created_year`, kèm đường `pct_scored` trên trục phụ bên phải.

In [ ]:
df_year = q(
    """
    SELECT
        created_year,
        n_products,
        pct_scored
    FROM mart.products_by_year
    ORDER BY created_year
    """
)

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.bar(df_year["created_year"], df_year["n_products"], color="#4C72B0", label="Số sản phẩm")
ax1.set_xlabel("Năm tạo (created_year)")
ax1.set_ylabel("Số sản phẩm")
ax1.set_title("Số sản phẩm theo năm tạo và tỷ lệ có Nutri-Score")

ax2 = ax1.twinx()
ax2.plot(
    df_year["created_year"],
    df_year["pct_scored"],
    color="#C44E52",
    marker="o",
    label="% có Nutri-Score",
)
ax2.set_ylabel("% có Nutri-Score")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

fig.tight_layout()
fig.savefig(img_dir / "products_by_year.png", dpi=150)
plt.show()

### Insight
